<a href="https://colab.research.google.com/github/kumarsirish/agentic-rag-workshop/blob/main/Movie-RAG-Basics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Understanding our Movie RAG Pipeline

This notebook demonstrates a complete, lightweight **Retrieval-Augmented Generation (RAG)** pipeline designed to recommend movies from our private dataset.

### 1. Vector Search (The Retrieval Step)
Instead of doing a keyword match (like searching for the word "space"), we use **semantic search**:
* **Embeddings (`SentenceTransformer`):** We turn our movie plots into vectors (dense arrays of 384 numbers) representing their core concepts and meanings.
* **ChromaDB:** We store these vectors and their metadata in an in-memory ChromaDB collection, allowing us to find similar items in microseconds using vector similarity query logic.

### 2. Context Integration (The Augment Step)
* **Prompt Construction (`build_prompt`):** Once we find the top closest movies to the user's query, we structure their full information into a string template. This limits the context window of our generative model to *only* the movies we retrieved, mitigating model hallucination.

### 3. Generation (The GenAI Step)
* **Large Language Model (`gemini-2.5-flash`):** The LLM receives the enriched context prompt and answers the user's question naturally, synthesizing recommendations specifically from the provided data structure.

# Movie Recommendation with RAG: the basics

**RAG = Retrieval-Augmented Generation.** Instead of asking an LLM to answer from memory, we first *look up* the facts we trust, then let the LLM answer *using only those facts*.

```
 Your question
      |
      v
 1. RETRIEVE   find the movies closest in meaning to the question      (embeddings + Chromadb)
      |
      v
 2. AUGMENT    paste those movies into a prompt                         (plain Python string)
      |
      v
 3. GENERATE   the LLM writes the answer from that prompt               (Gemini)
```

We will build it in small steps, one function per step. Each cell is short, so run them one at a time and read the output.

## 0. Setup

In [1]:
# Run once to install our dependencies.
%pip install -q sentence-transformers google-genai gradio chromadb

In [2]:
import os
import json
import urllib.parse
import urllib.request

from sentence_transformers import SentenceTransformer
from google import genai
from IPython.display import display, HTML

# Settings
EMBEDDING_MODEL = "all-MiniLM-L6-v2"     # turns text into 384 numbers
LLM_MODEL = "gemini-2.5-flash"           # writes the final answer
TOP_K = 5                                # how many movies to retrieve

In [3]:
import os
from google.colab import userdata

try:
    # Read the GOOGLE_API_KEY securely from the Colab secrets vault
    os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
except Exception as e:
    print("Could not retrieve GOOGLE_API_KEY from Colab secrets.")
    print("Please ensure you have added GOOGLE_API_KEY to your Colab secrets vault (the 🔑 icon on the left).")

try:
    # Read the GOOGLE_API_KEY securely from the Colab secrets vault
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception as e:
    print("Could not retrieve HF_TOKEN from Colab secrets.")
    print("Please ensure you have added HF_TOKEN to your Colab secrets vault (the 🔑 icon on the left).")

print("Google API key found in environment:", "GOOGLE_API_KEY" in os.environ)
print("HF_TOKEN  key found in environment:", "HF_TOKEN" in os.environ)

Google API key found in environment: True
HF_TOKEN  key found in environment: True


## 1. Our knowledge base

RAG searches *our own data*. Here it is a small list of 32 movies. Each movie is a Python dictionary.
(Ratings are approximate, and the one-line reviews were written for this demo. They are not real quotes.)

In [4]:
movies = [
    {"title": "Inception", "year": 2010, "genre": "Sci-Fi, Thriller", "rating": 8.8,
     "cast": "Leonardo DiCaprio, Joseph Gordon-Levitt, Elliot Page",
     "overview": "A thief who steals secrets by entering people's dreams is hired to plant an idea in a target's mind.",
     "review": "Mind-bending and beautifully shot; you will want to watch it twice."},
    {"title": "Interstellar", "year": 2014, "genre": "Sci-Fi, Drama", "rating": 8.7,
     "cast": "Matthew McConaughey, Anne Hathaway, Jessica Chastain",
     "overview": "Astronauts travel through a wormhole to find a new home for humanity as Earth's crops fail.",
     "review": "Epic and emotional, with a score that stays with you."},
    {"title": "The Matrix", "year": 1999, "genre": "Sci-Fi, Action", "rating": 8.7,
     "cast": "Keanu Reeves, Laurence Fishburne, Carrie-Anne Moss",
     "overview": "A hacker discovers that reality is a simulation and joins a rebellion against the machines.",
     "review": "Groundbreaking action and a story that makes you question reality."},
    {"title": "Toy Story", "year": 1995, "genre": "Animation, Family", "rating": 8.3,
     "cast": "Tom Hanks, Tim Allen, Don Rickles",
     "overview": "A cowboy doll feels threatened when a new space ranger toy arrives in a boy's room.",
     "review": "Warm, funny and full of heart for kids and adults alike."},
    {"title": "Finding Nemo", "year": 2003, "genre": "Animation, Family", "rating": 8.2,
     "cast": "Albert Brooks, Ellen DeGeneres, Alexander Gould",
     "overview": "A timid clownfish crosses the ocean to rescue his son.",
     "review": "A colorful, touching adventure about a father's love."},
    {"title": "The Shawshank Redemption", "year": 1994, "genre": "Drama", "rating": 9.3,
     "cast": "Tim Robbins, Morgan Freeman, Bob Gunton",
     "overview": "A banker wrongly jailed for murder finds hope and friendship over decades in prison.",
     "review": "A quiet, powerful story about hope that lifts you up."},
    {"title": "The Godfather", "year": 1972, "genre": "Crime, Drama", "rating": 9.2,
     "cast": "Marlon Brando, Al Pacino, James Caan",
     "overview": "The aging head of a crime family hands control of the empire to his reluctant son.",
     "review": "Slow-burning, masterful storytelling about family and power."},
    {"title": "The Dark Knight", "year": 2008, "genre": "Action, Crime", "rating": 9.0,
     "cast": "Christian Bale, Heath Ledger, Aaron Eckhart",
     "overview": "Batman faces the Joker, a chaotic criminal who pushes Gotham City to its limits.",
     "review": "Dark and intense, with an unforgettable villain."},
    {"title": "Forrest Gump", "year": 1994, "genre": "Drama, Romance", "rating": 8.8,
     "cast": "Tom Hanks, Robin Wright, Gary Sinise",
     "overview": "A kind man with a low IQ unknowingly takes part in major moments of American history.",
     "review": "Gentle, moving and quietly funny; a real feel-good classic."},
    {"title": "Parasite", "year": 2019, "genre": "Thriller, Drama", "rating": 8.5,
     "cast": "Song Kang-ho, Lee Sun-kyun, Cho Yeo-jeong",
     "overview": "A poor family schemes to get jobs in a wealthy household, with unexpected consequences.",
     "review": "Sharp, darkly funny and full of surprises."},
    {"title": "Spirited Away", "year": 2001, "genre": "Animation, Fantasy", "rating": 8.6,
     "cast": "Rumi Hiiragi, Miyu Irino, Mari Natsuki",
     "overview": "A girl wanders into a world of spirits and must work in a bathhouse to free her parents.",
     "review": "Imaginative and magical; every frame feels hand-crafted."},
    {"title": "The Lion King", "year": 1994, "genre": "Animation, Family", "rating": 8.5,
     "cast": "Matthew Broderick, Jeremy Irons, James Earl Jones",
     "overview": "A young lion prince flees after his father's death and later returns to reclaim his kingdom.",
     "review": "A timeless, emotional story with wonderful songs."},
    {"title": "Mad Max: Fury Road", "year": 2015, "genre": "Action, Sci-Fi", "rating": 8.1,
     "cast": "Tom Hardy, Charlize Theron, Nicholas Hoult",
     "overview": "In a desert wasteland, a drifter and a rebel driver flee a tyrant in one long chase.",
     "review": "Non-stop, breathtaking action with very little dialogue."},
    {"title": "La La Land", "year": 2016, "genre": "Romance, Musical", "rating": 8.0,
     "cast": "Ryan Gosling, Emma Stone, John Legend",
     "overview": "A jazz pianist and an aspiring actress fall in love while chasing their dreams in Los Angeles.",
     "review": "Dreamy and romantic with a bittersweet, memorable ending."},
    {"title": "The Grand Budapest Hotel", "year": 2014, "genre": "Comedy, Adventure", "rating": 8.1,
     "cast": "Ralph Fiennes, Tony Revolori, F. Murray Abraham",
     "overview": "A legendary hotel concierge and his young lobby boy are pulled into the theft of a priceless painting.",
     "review": "Quirky, stylish and laugh-out-loud funny."},
    {"title": "Whiplash", "year": 2014, "genre": "Drama, Music", "rating": 8.5,
     "cast": "Miles Teller, J.K. Simmons, Melissa Benoist",
     "overview": "A young drummer is pushed to his limits by a ruthless music teacher.",
     "review": "Tense and electrifying; you will be gripping your seat."},
    {"title": "Get Out", "year": 2017, "genre": "Horror, Thriller", "rating": 7.7,
     "cast": "Daniel Kaluuya, Allison Williams, Bradley Whitford",
     "overview": "A young man visiting his girlfriend's family slowly uncovers a disturbing secret.",
     "review": "Clever, creepy social thriller that keeps you guessing."},
    {"title": "Coco", "year": 2017, "genre": "Animation, Family", "rating": 8.4,
     "cast": "Anthony Gonzalez, Gael Garcia Bernal, Benjamin Bratt",
     "overview": "A boy who dreams of becoming a musician enters the Land of the Dead to uncover his family's history.",
     "review": "Colorful and full of music; keep tissues nearby."},
    {"title": "The Martian", "year": 2015, "genre": "Sci-Fi, Adventure", "rating": 8.0,
     "cast": "Matt Damon, Jessica Chastain, Chiwetel Ejiofor",
     "overview": "An astronaut stranded on Mars uses science and humor to survive until rescue.",
     "review": "Funny, optimistic and a joy for science lovers."},
    {"title": "Arrival", "year": 2016, "genre": "Sci-Fi, Drama", "rating": 7.9,
     "cast": "Amy Adams, Jeremy Renner, Forest Whitaker",
     "overview": "A linguist works to communicate with alien visitors before fear pushes the world toward war.",
     "review": "Thoughtful and moving; a slow-burn sci-fi that rewards patience."},
    {"title": "Back to the Future", "year": 1985, "genre": "Sci-Fi, Comedy", "rating": 8.5,
     "cast": "Michael J. Fox, Christopher Lloyd, Lea Thompson",
     "overview": "A teenager is sent thirty years into the past in a time-traveling car and must make sure his parents fall in love.",
     "review": "Pure fun: clever, fast and endlessly rewatchable."},
    {"title": "Amelie", "year": 2001, "genre": "Romance, Comedy", "rating": 8.3,
     "cast": "Audrey Tautou, Mathieu Kassovitz, Rufus",
     "overview": "A shy Parisian waitress secretly improves the lives of the people around her while searching for love.",
     "review": "Charming, whimsical and uplifting."},
    {"title": "Gladiator", "year": 2000, "genre": "Action, Drama", "rating": 8.5,
     "cast": "Russell Crowe, Joaquin Phoenix, Connie Nielsen",
     "overview": "A betrayed Roman general becomes a gladiator and seeks revenge on the corrupt emperor.",
     "review": "Grand, gritty and emotional with a powerful lead."},
    {"title": "The Silence of the Lambs", "year": 1991, "genre": "Thriller, Crime", "rating": 8.6,
     "cast": "Jodie Foster, Anthony Hopkins, Scott Glenn",
     "overview": "A young FBI trainee asks an imprisoned psychiatrist for help in catching a serial killer.",
     "review": "Chilling and suspenseful with unforgettable performances."},
    {"title": "Up", "year": 2009, "genre": "Animation, Adventure", "rating": 8.3,
     "cast": "Ed Asner, Jordan Nagai, Christopher Plummer",
     "overview": "An elderly widower ties balloons to his house and flies to South America with an unexpected young stowaway.",
     "review": "The opening alone will make you cry; the rest will make you smile."},
    {"title": "Knives Out", "year": 2019, "genre": "Mystery, Comedy", "rating": 7.9,
     "cast": "Daniel Craig, Chris Evans, Ana de Armas",
     "overview": "A detective investigates the death of a wealthy novelist surrounded by a family of suspects.",
     "review": "A witty, twisty whodunit that is great fun to guess along with."},
    {"title": "Everything Everywhere All at Once", "year": 2022, "genre": "Sci-Fi, Comedy", "rating": 7.8,
     "cast": "Michelle Yeoh, Stephanie Hsu, Ke Huy Quan",
     "overview": "A laundromat owner is pulled across parallel universes to stop a threat that could unravel reality.",
     "review": "Wild, inventive and surprisingly touching."},
    {"title": "The Pursuit of Happyness", "year": 2006, "genre": "Drama", "rating": 8.0,
     "cast": "Will Smith, Jaden Smith, Thandiwe Newton",
     "overview": "A struggling salesman and his young son face homelessness while he works toward a career in finance.",
     "review": "Inspiring and heartfelt, anchored by a warm father-son bond."},
    {"title": "Dune", "year": 2021, "genre": "Sci-Fi, Adventure", "rating": 8.0,
     "cast": "Timothee Chalamet, Rebecca Ferguson, Oscar Isaac",
     "overview": "A noble young heir travels to a dangerous desert planet to protect his family and its people.",
     "review": "Visually stunning and immersive on a big screen."},
    {"title": "Paddington 2", "year": 2017, "genre": "Family, Comedy", "rating": 7.8,
     "cast": "Ben Whishaw, Hugh Bonneville, Sally Hawkins",
     "overview": "A polite bear is wrongly jailed for a theft and his adoptive family works to prove his innocence.",
     "review": "Sweet, funny and kind-hearted; pure comfort viewing."},
    {"title": "Your Name", "year": 2016, "genre": "Animation, Romance", "rating": 8.4,
     "cast": "Ryunosuke Kamiki, Mone Kamishiraishi, Ryo Narita",
     "overview": "Two teenagers mysteriously swap bodies and try to find each other across distance and time.",
     "review": "Gorgeous animation and a romance that hits hard."},
    {"title": "Oppenheimer", "year": 2023, "genre": "Drama, History", "rating": 8.3,
     "cast": "Cillian Murphy, Emily Blunt, Matt Damon",
     "overview": "The story of the physicist who led the creation of the first atomic bomb and of its aftermath.",
     "review": "Dense, intense and gripping; a big, serious drama."},
]

print("Movies in our knowledge base:", len(movies))
movies[0]

Movies in our knowledge base: 32


{'title': 'Inception',
 'year': 2010,
 'genre': 'Sci-Fi, Thriller',
 'rating': 8.8,
 'cast': 'Leonardo DiCaprio, Joseph Gordon-Levitt, Elliot Page',
 'overview': "A thief who steals secrets by entering people's dreams is hired to plant an idea in a target's mind.",
 'review': 'Mind-bending and beautifully shot; you will want to watch it twice.'}

## 2. Turn each movie into text

The embedding model reads text, so we describe every movie as a string. We make **two** versions:

- `movie_to_text` is the full description. It is pasted into the prompt for the LLM.
- `movie_to_search_text` is a short version used for searching. It starts with the genre and plot, because that is what people usually ask about. Long text with cast names and reviews would dilute the meaning, and a short question like *"animated movies"* would then match the wrong movies.

We also write `Animation` as `Animation (animated cartoon)`, so a question that uses the word *animated* finds it.

In [5]:
def movie_to_text(movie):
    """Describe one movie as a single line of text."""
    return (f"{movie['title']} ({movie['year']}) | Genre: {movie['genre']} | Rating: {movie['rating']} "
            f"| Cast: {movie['cast']} | Plot: {movie['overview']} | Review: {movie['review']}")


def movie_to_search_text(movie):
    """A short description used only for searching: genre first, then the plot."""
    genre = movie["genre"].replace("Animation", "Animation (animated cartoon)")
    return f"{genre} movie. {movie['overview']}"


search_texts = []
for movie in movies:
    #search_texts.append(movie_to_search_text(movie))
    search_texts.append(movie_to_text(movie))

print(search_texts[3])

Toy Story (1995) | Genre: Animation, Family | Rating: 8.3 | Cast: Tom Hanks, Tim Allen, Don Rickles | Plot: A cowboy doll feels threatened when a new space ranger toy arrives in a boy's room. | Review: Warm, funny and full of heart for kids and adults alike.


## 3. Embeddings

An **embedding** is a list of numbers that captures the *meaning* of a text. Texts with similar meaning get similar numbers, so "a funny cartoon for kids" ends up close to *Toy Story*, even though they share no words.

**ChromaDB** stores all these vectors and quickly finds the closest ones. The first run downloads the model (about 90 MB).

In [6]:
embedder = SentenceTransformer(EMBEDDING_MODEL)
movie_vectors = embedder.encode(search_texts)

print("Shape:", movie_vectors.shape)                       # (32 movies, 384 numbers each)
print("First 5 numbers of movie 0:", movie_vectors[0][:5])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Shape: (32, 384)
First 5 numbers of movie 0: [-0.12738837 -0.04504668  0.02296315 -0.01150418  0.03554049]


In [8]:
import chromadb

# Initialize an in-memory ChromaDB client
chroma_client = chromadb.Client()

# Create or get a collection for our movies
# ChromaDB can automatically handle embeddings, but since we already have 'movie_vectors',
# we will pass our pre-computed sentence-transformer embeddings directly.
collection = chroma_client.get_or_create_collection(name="movie_collection")

# Prepare the data for ChromaDB
ids = [str(i) for i in range(len(movies))]
embeddings = movie_vectors.tolist() # ChromaDB expects a list of floats
metadatas = [
    {
        "title": m["title"],
        "year": m["year"],
        "genre": m["genre"],
        "rating": m["rating"],
        "cast": m["cast"],
        "overview": m["overview"],
        "review": m["review"]
    }
    for m in movies
]
documents = search_texts

# Add the movies to the collection
collection.add(
    embeddings=embeddings,
    documents=documents,
    metadatas=metadatas,
    ids=ids
)

print(f"Successfully added {collection.count()} movies to ChromaDB!")

Successfully added 32 movies to ChromaDB!


## 4. Retrieve (the **R** in RAG)

To search, we embed the *question* with the same model, then search ChromaDB for the closest movie vectors.
A **smaller distance means a closer match**.

In [9]:
def search_movies_chroma(question, k=TOP_K):
    """Return the k movies whose meaning is closest to the question using ChromaDB."""
    # 1. Embed the user's question
    question_vector = embedder.encode([question]).tolist()[0]

    # 2. Query the collection
    results = collection.query(
        query_embeddings=[question_vector],
        n_results=k
    )

    # 3. Format results to match the original structure
    found = []
    if results["metadatas"]:
        # Chroma results are nested because we can query multiple inputs at once
        for metadata, distance in zip(results["metadatas"][0], results["distances"][0]):
            movie = metadata.copy()
            movie["distance"] = round(float(distance), 3)
            found.append(movie)
    return found

def show_movies(found):
    """Print the movies as simple info cards."""
    for movie in found:
        print(f"{movie['title']} ({movie['year']})  |  Rating: {movie['rating']}  |  {movie['genre']}  |  distance {movie['distance']}")
        print(f"   Cast:    {movie['cast']}")
        print(f"   Summary: {movie['overview']}")
        print(f"   Review:  {movie['review']}")
        print()

# Test the new search function
question = "a funny animated movie the whole family can enjoy"
show_movies(search_movies_chroma(question, 3))

Up (2009)  |  Rating: 8.3  |  Animation, Adventure  |  distance 1.248
   Cast:    Ed Asner, Jordan Nagai, Christopher Plummer
   Summary: An elderly widower ties balloons to his house and flies to South America with an unexpected young stowaway.
   Review:  The opening alone will make you cry; the rest will make you smile.

Paddington 2 (2017)  |  Rating: 7.8  |  Family, Comedy  |  distance 1.323
   Cast:    Ben Whishaw, Hugh Bonneville, Sally Hawkins
   Summary: A polite bear is wrongly jailed for a theft and his adoptive family works to prove his innocence.
   Review:  Sweet, funny and kind-hearted; pure comfort viewing.

Everything Everywhere All at Once (2022)  |  Rating: 7.8  |  Sci-Fi, Comedy  |  distance 1.355
   Cast:    Michelle Yeoh, Stephanie Hsu, Ke Huy Quan
   Summary: A laundromat owner is pulled across parallel universes to stop a threat that could unravel reality.
   Review:  Wild, inventive and surprisingly touching.



## 5. Augment (the **A** in RAG)

"Augment" just means *adding the retrieved movies to the prompt*. We also tell the LLM to use **only** those movies. That instruction is what keeps the answer grounded in our data.

In [10]:
def build_prompt(question, found):
    """Put the retrieved movies and the question into one prompt."""
    movie_list = ""
    for movie in found:
        movie_list = movie_list + "- " + movie_to_text(movie) + "\n"

    return (
        "You are a friendly movie assistant.\n"
        "Answer the question using ONLY the movies listed below.\n"
        "If the user wants recommendations, pick the best 3 and give one short reason for each.\n"
        "If the movies below do not contain the answer, say so honestly.\n\n"
        f"Movies:\n{movie_list}\n"
        f"Question: {question}"
    )


question = "Something like Interstellar but more hopeful"
print(build_prompt(question, search_movies_chroma(question, 3)))

You are a friendly movie assistant.
Answer the question using ONLY the movies listed below.
If the user wants recommendations, pick the best 3 and give one short reason for each.
If the movies below do not contain the answer, say so honestly.

Movies:
- Interstellar (2014) | Genre: Sci-Fi, Drama | Rating: 8.7 | Cast: Matthew McConaughey, Anne Hathaway, Jessica Chastain | Plot: Astronauts travel through a wormhole to find a new home for humanity as Earth's crops fail. | Review: Epic and emotional, with a score that stays with you.
- Arrival (2016) | Genre: Sci-Fi, Drama | Rating: 7.9 | Cast: Amy Adams, Jeremy Renner, Forest Whitaker | Plot: A linguist works to communicate with alien visitors before fear pushes the world toward war. | Review: Thoughtful and moving; a slow-burn sci-fi that rewards patience.
- Oppenheimer (2023) | Genre: Drama, History | Rating: 8.3 | Cast: Cillian Murphy, Emily Blunt, Matt Damon | Plot: The story of the physicist who led the creation of the first atomic b

## 6. Generate (the **G** in RAG)

Now we send the prompt to the LLM. If there is no API key, the function says so instead of crashing, so the rest of the notebook still runs.

In [11]:
def ask_llm(prompt):
    client = genai.Client()
    response = client.models.generate_content(model=LLM_MODEL, contents=prompt)
    return response.text

## 7. The whole RAG pipeline

Three lines: retrieve, augment, generate.

In [12]:
def recommend(question):
    """RAG in three steps. Returns the answer and the movies that were retrieved using ChromaDB."""
    found = search_movies_chroma(question)      # 1. Retrieve (using ChromaDB)
    prompt = build_prompt(question, found)      # 2. Augment
    answer = ask_llm(prompt)                    # 3. Generate
    return answer, found

In [13]:
question = "Something like Interstellar, but shorter and more hopeful"

answer, found = recommend(question)

print("Question:", question)
print()
print("Retrieved movies:")
show_movies(found)
print("Answer:")
print(answer)

Question: Something like Interstellar, but shorter and more hopeful

Retrieved movies:
Interstellar (2014)  |  Rating: 8.7  |  Sci-Fi, Drama  |  distance 1.053
   Cast:    Matthew McConaughey, Anne Hathaway, Jessica Chastain
   Summary: Astronauts travel through a wormhole to find a new home for humanity as Earth's crops fail.
   Review:  Epic and emotional, with a score that stays with you.

Arrival (2016)  |  Rating: 7.9  |  Sci-Fi, Drama  |  distance 1.276
   Cast:    Amy Adams, Jeremy Renner, Forest Whitaker
   Summary: A linguist works to communicate with alien visitors before fear pushes the world toward war.
   Review:  Thoughtful and moving; a slow-burn sci-fi that rewards patience.

Oppenheimer (2023)  |  Rating: 8.3  |  Drama, History  |  distance 1.354
   Cast:    Cillian Murphy, Emily Blunt, Matt Damon
   Summary: The story of the physicist who led the creation of the first atomic bomb and of its aftermath.
   Review:  Dense, intense and gripping; a big, serious drama.

Bac

**Your turn:** change the question below and run the cell again. Try a genre, an actor, or a mood.

In [14]:
question = "Movies with Tom Hanks"

answer, found = recommend(question)

show_movies(found)
print(answer)

Forrest Gump (1994)  |  Rating: 8.8  |  Drama, Romance  |  distance 0.969
   Cast:    Tom Hanks, Robin Wright, Gary Sinise
   Summary: A kind man with a low IQ unknowingly takes part in major moments of American history.
   Review:  Gentle, moving and quietly funny; a real feel-good classic.

Toy Story (1995)  |  Rating: 8.3  |  Animation, Family  |  distance 1.095
   Cast:    Tom Hanks, Tim Allen, Don Rickles
   Summary: A cowboy doll feels threatened when a new space ranger toy arrives in a boy's room.
   Review:  Warm, funny and full of heart for kids and adults alike.

The Grand Budapest Hotel (2014)  |  Rating: 8.1  |  Comedy, Adventure  |  distance 1.227
   Cast:    Ralph Fiennes, Tony Revolori, F. Murray Abraham
   Summary: A legendary hotel concierge and his young lobby boy are pulled into the theft of a priceless painting.
   Review:  Quirky, stylish and laugh-out-loud funny.

Oppenheimer (2023)  |  Rating: 8.3  |  Drama, History  |  distance 1.251
   Cast:    Cillian Murphy, 

##Show the posters

This part is only for looks, it is not RAG. It looks up each poster on OMDb, then on TMDb, and shows a plain picture if neither has it.
Both services give free API keys (https://www.omdbapi.com/apikey.aspx and https://www.themoviedb.org/settings/api). You can use your own.

In [15]:
OMDB_KEY = "fc1fef96"
TMDB_KEY = "f5a0494cf81e7d1e1f13b65a953af255"


def read_json(url):
    """Download a web address and return its JSON content."""
    with urllib.request.urlopen(url, timeout=5) as response:
        return json.loads(response.read().decode())


def poster_from_omdb(title, year):
    """Return a poster url from OMDb, or None."""
    url = f"https://www.omdbapi.com/?t={urllib.parse.quote(title)}&y={year}&apikey={OMDB_KEY}"
    try:
        data = read_json(url)
        if data.get("Poster", "N/A").startswith("http"):
            return data["Poster"]
    except Exception as error:
        print("OMDb failed for", title, "->", error)
    return None


def poster_from_tmdb(title, year):
    """Return a poster url from TMDb, or None."""
    url = f"https://api.themoviedb.org/3/search/movie?api_key={TMDB_KEY}&query={urllib.parse.quote(title)}&year={year}"
    try:
        results = read_json(url).get("results", [])
        if len(results) > 0 and results[0].get("poster_path"):
            return "https://image.tmdb.org/t/p/w300" + results[0]["poster_path"]
    except Exception as error:
        print("TMDb failed for", title, "->", error)
    return None


def get_poster_url(title, year):
    """Try OMDb, then TMDb, then a plain placeholder picture."""
    poster = poster_from_omdb(title, year)
    if poster is None:
        poster = poster_from_tmdb(title, year)
    if poster is None:
        poster = "https://placehold.co/180x260/0f172a/ffffff?text=" + urllib.parse.quote(title)
    return poster


def movies_to_html(found):
    """Build one HTML card per movie, with a larger poster on the left."""
    cards = ""
    for movie in found:
        poster = get_poster_url(movie["title"], movie["year"])
        cards = cards + f"""
        <div style="display: flex; margin-bottom: 16px; border: 1px solid #ddd; padding: 12px; border-radius: 8px; background: #f9f9f9; color: #222;">
            <img src="{poster}" alt="{movie['title']} poster" style="width: 180px; margin-right: 18px; border-radius: 6px; object-fit: cover;"
                 onerror="this.src='https://placehold.co/180x260?text=No+Image'">
            <div>
                <h3 style="margin: 0; font-size: 1.35em;">{movie['title']} ({movie['year']})</h3>
                <p style="margin: 6px 0;"><b>Rating:</b> ⭐ {movie['rating']} | <b>Genre:</b> {movie['genre']}</p>
                <p style="margin: 6px 0;"><b>Cast:</b> {movie['cast']}</p>
                <p style="margin: 6px 0;">{movie['overview']}</p>
                <p style="margin: 6px 0; font-style: italic; color: #555;">\"{movie['review']}\"</p>
            </div>
        </div>
        """
    return cards

display(HTML(movies_to_html(search_movies_chroma("a funny animated movie the whole family can enjoy", 3))))

TMDb failed for Paddington 2 -> HTTP Error 401: Unauthorized


## Tiny web app with Gradio

The same `recommend` function behind a text box. Run the cell and use the app right here, or open the public link it prints.

In [16]:
import gradio as gr


def web_search(question):
    """What the web app runs when someone presses Submit."""
    answer, found = recommend(question)
    return answer, movies_to_html(found)


demo = gr.Interface(
    fn=web_search,
    inputs=gr.Textbox(label="What kind of movie do you want?", placeholder="e.g. a heartwarming animated movie for the whole family"),
    outputs=[gr.Markdown(label="Answer"), gr.HTML(label="Movies")],
    title="🎬 Movie RAG Recommender",
)

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://24cb0f74fe625ece6c.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## What to try next

- **Bigger data:** load a movie CSV (for example from Kaggle) into the same list-of-dictionaries format.
- **Different `TOP_K`:** retrieve 2 movies, then 10. What happens to the answer?
- **Edit the prompt:** ask for one sentence, or for a table. Retrieval stays the same, only the generation changes.
- **Ask something our data cannot answer** (for example *"movies about Titanic"*). Does the assistant admit it?
- **Make it agentic:** let an agent decide *when* to search and rewrite a weak query. That is the next part of the session (LangGraph, Google ADK, CrewAI).